# Titanic Survival Analysis — ABRAR JAWAD — May 2026

Problem Statement — 
1. Overall survival rate — what % survived?
2. Survival by Sex — bar chart + chi-square test 
3. Survival by Pclass — bar chart + chi-square test
4. Age distribution — histogram, compare survivors vs non-survivors; t-test on mean age
5. Fare vs Survival — boxplot
6. Correlation heatmap — numeric columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
df = pd.read_csv('../data/raw/train.csv')
print(df.isnull().sum())

In [ ]:
print(df.shape)        # how many rows and columns
df.head()              # first 5 rows

In [ ]:
df.describe()          # summary statistics for all numeric columns

### Part 1: Overall survival rate — what % survived?

In [ ]:
# Mean of a 0/1 column directly gives the proportion of 1s
survival_rate = df['Survived'].mean()
print(f"Survival rate: {survival_rate:.2%}")

Overall survival rate was 38.38% — meaning 61.62% of passengers did not survive.

### Part 2: Survival by Sex — bar chart + chi-square test 

In [ ]:
# Split-apply-combine: group by sex, compute survival rate per group
survival_by_sex = df.groupby('Sex')['Survived'].mean()
print(survival_by_sex)

In [ ]:
plt.figure(figsize=(6,4))
survival_by_sex.plot(kind='bar', color=['salmon', 'steelblue'], edgecolor='black')
plt.title('Survival Rate by Sex')
plt.xlabel('Sex')
plt.ylabel('Survival Rate')
plt.xticks(rotation=0)
plt.ylim(0,1)
plt.tight_layout()
plt.savefig('../visuals/survival_by_sex.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Build observed contingency table — required input for chi-square test
contingency = pd.crosstab(df['Sex'], df['Survived'])
print(contingency)

chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f"chi2 stat : {chi2:.2f}")
print(f"p-value : {p:.4f}")
print(f"degrees of freedom : {dof}")

In [ ]:
if (p<0.05):
    print("Sex and survival are significantly associated.")
else:
    print("No significant association between sex and survival.")

Chi-square test: χ²=260.72, p≈0.000, dof=1
Sex and survival are NOT independent.
Females had a significantly higher survival rate than males.
This aligns with the "women and children first" evacuation protocol.

### Part 3: Survival by Pclass — bar chart + chi-square test 

In [ ]:
survival_by_pclass = df.groupby('Pclass')['Survived'].mean()
print(survival_by_pclass)

In [ ]:
plt.figure(figsize=(6,4))
survival_by_pclass.plot(kind='bar', color=['gold', 'silver', 'peru'], edgecolor='black')
plt.title('Survival Rate by Passenger Class')
plt.xlabel('Passenger Class')
plt.ylabel('Survival Rate')
plt.xticks(rotation=0)
plt.ylim(0,1)
plt.tight_layout()
plt.savefig('../visuals/survival_by_pclass.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
contingency2 = pd.crosstab(df['Pclass'], df['Survived'])
print(contingency2)

chi2, p, dof, expected = stats.chi2_contingency(contingency2)
print(f"chi2 stat : {chi2:.2f}")
print(f"p-value : {p:.4f}")
print(f"degrees of freedom : {dof}")

In [ ]:
if (p<0.05):
        print("Passenger Class and survival are significantly associated.")
else:
        print("No significant association between Passenger class and survival.")

Chi-square test: χ²=102.89, p≈0.000, dof=2
Passenger class and survival are NOT independent.
Class 1 passengers had a significantly higher survival rate than the other two classes.
The likely reasons: 1st class cabins were on upper decks closer to lifeboats, and there's historical evidence of priority given to wealthier passengers during evacuation.

### Part 4: Age distribution — histogram, compare survivors vs non-survivors; t-test on mean age

In [ ]:
# dropna() because 177 age values are missing — can't plot NaN
survived_age = df[df['Survived'] == 1]['Age'].dropna()
not_survived_age = df[df['Survived'] == 0]['Age'].dropna()

In [ ]:
print(f"Survivors: {len(survived_age)} passengers, mean age: {survived_age.mean():.2f}")
print(f"Not survived: {len(not_survived_age)} passengers, mean age = {not_survived_age.mean():.2f}")


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(survived_age,  bins=30, alpha=0.5, color='lightcoral', label='Survived')
plt.hist(not_survived_age, bins=30, alpha=0.5, color='steelblue', label='Not Survived')
plt.title("Age Distribution: Survivors vs Non-Survivors")
plt.xlabel('Age')
plt.ylabel('Number of Passengers')
plt.legend()
plt.tight_layout()
plt.savefig("../visuals/age_distribution.png",  dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
t_stat, p_val = stats.ttest_ind(survived_age, not_survived_age)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val:.4f}")

In [ ]:
if p_val < 0.05:
        print("Age significantly differs between survivors and non-survivors (p < 0.05).")
else:
        print("No significant age difference found.")

**t-Test Result:** t = −2.0667, p = 0.0391, two-sample independent t-test

The negative t-statistic indicates that survivors had a lower mean age than 
non-survivors — survivors were on average approximately 2 years younger. With 
p = 0.039 < 0.05, we reject H₀ and conclude that the mean age difference is 
statistically significant.

**However, significance does not mean strength.** Compare this result to Q2 
(p ≈ 0.000) and Q3 (p ≈ 0.000). The age effect, while real, sits at the edge 
of the significance threshold — making it the weakest confirmed predictor in 
this analysis.

**Histogram Observation:** Both distributions peak between ages 20–35, reflecting 
the general passenger demographic. The most meaningful visual signal is in the 
0–10 bin, where survivor counts visibly exceed non-survivor counts — consistent 
with child prioritisation during evacuation. The broad overlap of the two 
distributions visually confirms that age alone was not a strong differentiator 
of survival outcome.

**Missing Data Limitation:** 177 passengers (≈20% of the dataset) had no recorded 
age and were excluded from this analysis via `.dropna()`. If these missing records 
are not randomly distributed across passenger groups — for example, if 3rd class 
passengers were disproportionately unrecorded — the sample used here may be 
slightly biased, and the mean estimates should be interpreted with that caveat.

**Conclusion:** Age was a statistically significant but weak predictor of survival. 
The child survival signal is present but modest. Sex (Q2) and passenger class (Q3) 
remain the dominant predictors identified so far.

### Part 5: Fare vs Survival — boxplot

In [ ]:
print(df[['Fare', 'Survived']].info())

In [ ]:
clean_fare = df.dropna(subset='Fare')
survived_fare = clean_fare[clean_fare['Survived']==1]['Fare']
not_survived_fare = clean_fare[clean_fare['Survived']==0]['Fare']

In [ ]:
plt.figure(figsize=(8,5))
plt.boxplot([survived_fare, not_survived_fare], labels=['Survived', 'Not Survived'], patch_artist=True, boxprops=dict(facecolor='lightblue', color='black'), medianprops=dict(color='red'))
plt.yscale('log')  # Log scale to handle skewness in fare distribution
plt.grid(axis='y', linestyle='--', alpha=0.7)  # Add grid for better readability
plt.title('Distribution of Fare by Survival')
plt.xlabel('Survival Status')
plt.ylabel('Fare')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../visuals/fare_boxplot.png", dpi=150, bbox_inches='tight')    
plt.show()

median_0 = clean_fare[clean_fare['Survived']==0]['Fare'].median()
median_1 = clean_fare[clean_fare['Survived']==1]['Fare'].median()
print(f"Median Fare for Not Survived: {median_0:.2f}")
print(f"Median Fare for Survived: {median_1:.2f}")

**Median Disparity**: Survivors paid 148% more than non-survivors at the median ($26.00 vs $10.50)

**Distribution Shape**: 
   - Non-survivors' fares are concentrated at the low end ($7-30 range)
   - Survivors show a broader distribution, including a significant presence in mid-range ($30-100) and high-end (>$100) fares

**Outliers**: Both groups contain high-fare outliers (>$200), confirming that wealth did not guarantee survival—some first-class passengers perished

**Limitations**

- **Causality**: Fare is a proxy variable. The true causal factors include cabin location, class, and social priority
- **Confounding**: Gender and age (not shown here) likely interact with fare to influence survival
- **Currency**: 1912 USD; historical conversion not applied ($1 ≈ $30-35 today)

The analysis demonstrates a statistically significant relationship between fare paid and survival likelihood. This serves as quantitative evidence of socioeconomic stratification in disaster response—a pattern still relevant in modern emergency management studies.

### Part 6: Correlation heatmap — numeric columns

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
numeric_df = numeric_df.drop(columns=['PassengerId'])
numeric_df['Age']=numeric_df['Age'].fillna(numeric_df['Age'].median())
correlation_matrix = numeric_df.corr()
plt.figure(figsize=(10,8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, cbar_kws={'shrink': 0.8}, center=0, square=True)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.savefig("../visuals/correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print("=== Correlation Analysis Results ===\n")
print("Correlation with Survival:")
print(correlation_matrix['Survived'].sort_values(ascending=False))
print("\n")

corr_values = correlation_matrix.unstack().sort_values(ascending=False)
corr_values = corr_values[corr_values < 0.999]
print("Top Correlations (excluding 1.0):")
print(corr_values.head(10))
print("\n")

print("Top Negative Correlations:")
print(corr_values.tail(10))

## Conclusions & Key Insights

### What this analysis set out to answer
> Which passenger characteristics most strongly predicted survival on the Titanic?

---

### Finding 1 — Sex was the dominant predictor
Females survived at **74.2%** vs **18.9%** for males — a 55 percentage point gap.  
Chi-square test: χ²=260.72, p≈0.000, dof=1. Critical value at 5% = 3.84.  
This is the most statistically decisive result in the entire analysis.  
Consistent with the documented "women and children first" evacuation protocol.

### Finding 2 — Passenger class showed a strong socioeconomic gradient
Survival rates: **1st class 63.0%** → **2nd class 47.3%** → **3rd class 24.2%**.  
Chi-square test: χ²=102.89, p≈0.000, dof=2. Critical value at 5% = 5.99.  
Probable causes: 1st class cabins were on upper decks closer to lifeboats, and 
historical accounts document preferential treatment of wealthier passengers.

### Finding 3 — Fare reflected the same socioeconomic pattern
Median fare for survivors: **£26.00** vs **£10.50** for non-survivors — a **148% gap**.  
Fare and Pclass are strongly correlated (r=−0.55 in the heatmap), confirming they 
measure the same underlying factor: wealth. Fare is not an independent predictor — 
it is a proxy for class.

### Finding 4 — Age had a weak but statistically significant effect
Survivors were on average ~2 years younger than non-survivors.  
Two-sample t-test: t=−2.07, p=0.039. Just below the 0.05 threshold.  
Compare this to Sex (p≈0.000) — age was a real but much weaker factor.  
The histogram shows slightly more child survivors in the 0–10 bin, consistent 
with child prioritization.

---

### Overall Conclusion
Three factors drove survival on the Titanic: **gender, class, and wealth** — all 
strongly intercorrelated. The data statistically confirms what historical accounts 
describe: women and children from upper classes had the highest chance of survival.

---

### Limitations
- This is **exploratory analysis only** — correlation does not imply causation
- Sex, Pclass, and Fare are intercorrelated — their individual effects cannot be cleanly separated without a multivariate model
- 177 passengers had no recorded age — excluded from Q4, which may introduce bias
- Cabin data (~77% missing) was excluded entirely and may have contained location-based survival signal

---

### What comes next
The natural next step is building a **logistic regression classifier** to predict survival probability using these features together — exactly what the Kaggle competition is designed for.